# Preparazione e Caricamento del Dataset VQA-RAD

Questo notebook gestisce l'intera pipeline di preparazione del dataset **VQA-RAD** (Visual Question Answering in Radiology), un dataset pubblico pensato per il VQA in ambito radiologico. Il punto di partenza è il dataset grezzo — un file JSON con le annotazioni e una cartella di immagini mediche — e l'obiettivo finale è produrre un dataset strutturato, diviso in split e disponibile su HuggingFace Hub per essere caricato in qualsiasi ambiente di training.

Il flusso completo è:
1. **Configurazione** dei percorsi locali
2. **Parsing** del JSON e costruzione del Dataset HuggingFace con immagini embedded
3. **Split** stratificato 70% / 15% / 15% (train / validation / test)
4. **Upload** su HuggingFace Hub
5. **Verifica** del dataset salvato

## Configurazione dei percorsi

Prima di tutto definiamo le tre variabili di percorso che il notebook utilizzerà durante tutta la sua esecuzione. `JSON_PATH` punta al file con le annotazioni del dataset, `IMAGES_DIR` alla cartella con le immagini radiologiche e `OUTPUT_DIR` alla destinazione dove verrà salvato il dataset processato in formato HuggingFace.

Tenerli tutti raggruppati in cima al notebook è una scelta pratica: permette di adattare facilmente il codice a macchine o percorsi diversi senza dover modificare nulla nel resto del codice.

In [1]:
# ── Percorsi ─────────────────────────────────────────────────────────────
JSON_PATH  = r"C:\\Users\\angel\\OneDrive\\Desktop\\ProgettoNLP\\progetto\\dataset\\VQA_RAD Dataset Public.json"
IMAGES_DIR = r"C:\\Users\\angel\\OneDrive\\Desktop\\ProgettoNLP\\progetto\\dataset\\VQA_RAD Image Folder"
OUTPUT_DIR = r"C:\\Users\\angel\\OneDrive\\Desktop\\ProgettoNLP\\progetto\\dataset\\VQA_RAD Output Folder"

## Costruzione del Dataset HuggingFace

Questo è il blocco centrale del notebook. Partiamo caricando il file JSON con i 2248 record del dataset VQA-RAD, ciascuno contenente la domanda clinica, la risposta, il tipo di risposta, l'organo raffigurato e il nome del file immagine corrispondente.

Per ogni record apriamo l'immagine dal disco e la convertiamo in RGB, registrando eventuali file mancanti senza interrompere il processo. Ogni riga viene raccolta in un dizionario con tutti i campi rilevanti.

Una volta costruita la lista completa, creiamo un `Dataset` HuggingFace definendo esplicitamente lo schema con `Features`: il campo `image` viene tipizzato come `HFImage()`, permettendo alla libreria di gestire le immagini in modo nativo e ottimizzato (encoding Arrow con lazy decoding).

Infine dividiamo il dataset in tre parti: **70% training, 15% validation, 15% test**, usando un seed fisso (42) per garantire che gli split siano sempre riproducibili. Il dataset viene poi salvato su disco in formato Arrow, pronto per l'upload.

In [ ]:
import json
from pathlib import Path
from PIL import Image
from datasets import Dataset, DatasetDict, Features, Value, Image as HFImage


# ── Carica JSON ───────────────────────────────────────────────────────────
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Record nel JSON: {len(data)}")

# ── Costruisci le righe ───────────────────────────────────────────────────
images_path = Path(IMAGES_DIR)
rows = []
missing = []

for rec in data:
    img_name = rec.get("image_name")
    img_path = images_path / img_name if img_name else None

    if img_path and img_path.exists():
        pil_img = Image.open(img_path).convert("RGB")
    else:
        missing.append(img_name)
        pil_img = None

    rows.append({
        "qid":            str(rec.get("qid", "")),
        "image_name":     img_name,
        "image":          pil_img,
        "image_organ":    rec.get("image_organ"),
        "question":       rec.get("question"),
        "question_type":  rec.get("question_type"),
        "phrase_type":    rec.get("phrase_type"),
        "answer":         rec.get("answer"),
        "answer_type":    rec.get("answer_type"),
    })

print(f"Immagini mancanti: {len(missing)}")

# ── Crea Dataset ──────────────────────────────────────────────────────────
cols = {key: [row[key] for row in rows] for key in rows[0].keys()}

features = Features({
    "qid":           Value("string"),
    "image_name":    Value("string"),
    "image":         HFImage(),
    "image_organ":   Value("string"),
    "question":      Value("string"),
    "question_type": Value("string"),
    "phrase_type":   Value("string"),
    "answer":        Value("string"),
    "answer_type":   Value("string"),
})

ds = Dataset.from_dict(cols, features=features)
print(f"\nDataset creato: {ds}")

# ── Split train / validation / test (70 / 15 / 15) ───────────────────────
split_tv = ds.train_test_split(test_size=0.30, seed=42)
split_vt = split_tv["test"].train_test_split(test_size=0.50, seed=42)

dataset_dict = DatasetDict({
    "train":      split_tv["train"],
    "validation": split_vt["train"],
    "test":       split_vt["test"],
})

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
dataset_dict.save_to_disk(OUTPUT_DIR)
print(f"\nDataset salvato in: {OUTPUT_DIR}")
print(f"  train:      {len(dataset_dict['train'])} esempi")
print(f"  validation: {len(dataset_dict['validation'])} esempi")
print(f"  test:       {len(dataset_dict['test'])} esempi")

Record nel JSON: 2248
Immagini mancanti: 0

Dataset creato: Dataset({
    features: ['qid', 'image_name', 'image', 'image_organ', 'question', 'question_type', 'phrase_type', 'answer', 'answer_type'],
    num_rows: 2248
})


Saving the dataset (1/1 shards): 100%|██████████| 338/338 [00:00<00:00, 821.11 examples/s]


Dataset salvato in: C:\Users\angel\OneDrive\Desktop\ProgettoNLP\progetto\dataset\VQA_RAD Output FolderValidation
  train:      1573 esempi
  validation: 337 esempi
  test:       338 esempi


## Upload su HuggingFace Hub

Il dataset è ora pronto e salvato localmente. Effettuiamo il login a HuggingFace tramite `login()` — che gestisce l'autenticazione con il token personale — e poi usiamo `upload_folder` per caricare l'intera cartella sul repository `Angelo0102/VQA-RAD`.

Questo passaggio è fondamentale per poter accedere al dataset negli ambienti di training remoti come Kaggle o Google Colab, senza dover trasferire manualmente i file ogni volta. Una volta caricato, il dataset sarà recuperabile con `load_from_disk` o `load_dataset` da qualsiasi macchina autenticata.

In [5]:
from huggingface_hub import login, upload_folder

login()
upload_folder(folder_path=OUTPUT_DIR, repo_id="Angelo0102/VQA-RAD", repo_type="dataset")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Angelo0102/VQA-RAD/commit/6547d9bd30d3d0bd0807ce6b9e3702accd749478', commit_message='Upload folder using huggingface_hub', commit_description='', oid='6547d9bd30d3d0bd0807ce6b9e3702accd749478', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Angelo0102/VQA-RAD', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Angelo0102/VQA-RAD'), pr_revision=None, pr_num=None)

## Verifica del dataset salvato

Un rapido sanity-check per assicurarsi che tutto sia andato a buon fine. Ricarichiamo il dataset dal disco con `load_from_disk`, accediamo al primo esempio dello split di training e verifichiamo tre cose: il tipo dell'oggetto immagine (deve essere un'immagine PIL), la domanda testuale e la risposta associata.

Se la stampa mostra un oggetto PIL e dei testi leggibili, significa che la pipeline di preparazione ha funzionato correttamente end-to-end, dalle immagini raw al dataset strutturato pronto per essere usato nel training.

In [6]:
from datasets import load_from_disk
ds = load_from_disk(OUTPUT_DIR)

sample = ds["train"][0]
print(type(sample["image"]))   
sample["image"].show()          
print(sample["question"])
print(sample["answer"])

<class 'PIL.PngImagePlugin.PngImageFile'>
Is there a fracture?
No
